# NLP Preprocessing Engine
### Data Science Internship – Feb 2026 | Task 1

Building a text preprocessing pipeline that can clean messy real world data.  
Going step by step through each task.

## Setting up — installing and importing what we need

In [2]:
# installing emoji library since it doesnt come by default
!pip install emoji --quiet


[notice] A new release of pip available: 22.2.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import re
import emoji
from collections import Counter

print("libraries loaded")

libraries loaded


---

## Task 1 — Conceptual Questions

**Q1. What is the difference between "Love" and "love" in NLP?**

So basically in NLP every token is matched exactly the way it appears. That means "Love" and "love" are treated as two completely separate words even though they mean the same thing. If we dont lowercase the text, a model will learn them as different features which is wasteful and inaccurate. For example if "Love" appears 40 times in headlines and "love" appears 200 times in normal text, the model splits that signal in two instead of combining it. Thats why lowercasing is one of the first steps we do in preprocessing — it makes sure the same word always maps to the same token regardless of where it appeared.

---

**Q2. What happens if stopwords are not removed?**

Stopwords are extremely common words like "the", "is", "and", "a", "in" etc. If we keep them in the data:
- They will show up at the top of every frequency count and dominate the results
- They push actual meaningful words down in importance
- Models trained on this data end up learning that "the" and "is" are super important features, which is obviously wrong
- It also bloats the vocabulary size unnecessarily which slows everything down

Basically the signal to noise ratio drops a lot when stopwords are kept.

---

**Q3. Two real world scenarios where removing stopwords can actually be harmful:**

**Scenario 1 — Sentiment with negation:**  
Take this sentence: *"This product is not bad at all"*  
"not" is a stopword. If we remove it the sentence becomes *"product bad"* — which flips the entire meaning from positive to negative. In any sentiment analysis task, blindly removing stopwords like "not", "no", "never" will give completely wrong results.

**Scenario 2 — Search engines and question answering:**  
If someone searches *"who is the PM of India"* and we strip stopwords, it becomes *"PM India"*. Now the context of it being a "who" question is lost. The system might return generic articles about India rather than the specific answer. In search and QA systems, stopwords often carry the query intent.

---

**Q4. Stemming vs Lemmatization — whats the actual difference?**

Both try to reduce words to their root form but they do it very differently.

Stemming just chops off the end of the word using fixed rules — it doesnt care if the result is a real word or not. So "studies" becomes "studi" and "running" becomes "run" but "fairly" might become "fairli" which is not even a valid english word.

Lemmatization actually looks the word up in a dictionary and returns its proper base form called the lemma. So "studies" becomes "study", "running" becomes "run", "better" becomes "good". The output is always a real word.

Stemming is faster but sloppy. Lemmatization is slower but gives cleaner and more accurate results. For production NLP systems, lemmatization is usually preferred.

---

## Task 2 — Building the Preprocessing Function

I broke it into small helper functions so each step is easy to read, test and modify separately. Then the main `preprocess_text()` calls them in order.

In [5]:
# words that are short (<=2 chars) but should NOT be removed
# 'no' and 'not' change the entire meaning of a sentence so we keep them
KEEP_WORDS = {"no", "not"}


def remove_urls_emails(text):
    # removing http/https links first, then www links, then anything that looks like an email
    text = re.sub(r'http\S+|https\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    return text


def remove_emojis_from_text(text):
    # using the emoji library to strip out all emoji characters cleanly
    return emoji.replace_emoji(text, replace='')


def remove_numbers(text):
    # removing standalone numbers like '2', '100', '9876543210'
    # \b makes sure we only match whole number tokens not digits inside words
    text = re.sub(r'\b\d+\b', '', text)
    # also removing any leftover digits that might be stuck to symbols like '0/10'
    text = re.sub(r'\d+', '', text)
    return text


def remove_special_chars(text):
    # keeping only letters and spaces, everything else goes (punctuation, $, %, !, etc.)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text


def fix_repeated_chars(text):
    # if the same character repeats 3 or more times in a row, collapse it to just 1
    # eg: 'soooo' becomes 'so', 'goooood' becomes 'god'
    # this handles internet slang and exaggerated typing
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    return text


def to_lower(text):
    return text.lower()


def clean_spaces(text):
    # stripping leading/trailing spaces and collapsing multiple spaces into one
    return re.sub(r'\s+', ' ', text).strip()


def filter_short_tokens(tokens):
    # removing tokens with 2 or fewer characters
    # but keeping 'no' and 'not' because they affect meaning a lot
    filtered = []
    for word in tokens:
        if word in KEEP_WORDS:
            filtered.append(word)
        elif len(word) > 2:
            filtered.append(word)
        # else: skip it
    return filtered


print("helper functions ready")

helper functions ready


In [8]:
def preprocess_text(text):
    """
    Takes a raw string and runs it through the full cleaning pipeline.
    Returns: (list of tokens, cleaned sentence as string)

    Steps in order:
    1. check if input is valid
    2. remove URLs and emails
    3. remove emojis
    4. remove numbers
    5. remove punctuation and special chars
    6. collapse repeated characters
    7. lowercase everything
    8. fix extra whitespace
    9. split into tokens
    10. drop very short tokens (except no/not)
    """

    # step 1 — handling edge cases upfront so rest of code doesnt break
    if not isinstance(text, str):
        return [], ""
    if text.strip() == "":
        return [], ""

    # step 2
    text = remove_urls_emails(text)

    # step 3
    text = remove_emojis_from_text(text)

    # step 4
    text = remove_numbers(text)

    # step 5
    text = remove_special_chars(text)

    # step 6 — doing this before lowercase so casing doesnt interfere
    text = fix_repeated_chars(text)

    # step 7
    text = to_lower(text)

    # step 8
    text = clean_spaces(text)

    # step 9 — simple whitespace split is fine after all the cleaning above
    tokens = text.split()

    # step 10
    tokens = filter_short_tokens(tokens)

    clean_sentence = ' '.join(tokens)
    return tokens, clean_sentence


print("preprocess_text() is ready")

preprocess_text() is ready


In [10]:
# quick check to make sure each feature is working correctly before stress testing
print("--- Feature checks ---\n")

# numbers
t, s = preprocess_text("I have 2 dogs")
print(f"Remove numbers     : {s}")

# extra spaces
t, s = preprocess_text("This  is   good")
print(f"Extra spaces       : {s}")

# repeated chars
t, s = preprocess_text("soooo goooood")
print(f"Repeated chars     : {s}")

# lowercase
t, s = preprocess_text("WOW This is GREAT")
print(f"Lowercase          : {s}")

# keep 'not'
t, s = preprocess_text("I am not ok")
print(f"Keep not/no        : {s}")

# url removal
t, s = preprocess_text("Visit http://example.com now")
print(f"URL removal        : {s}")

--- Feature checks ---

Remove numbers     : have dogs
Extra spaces       : this good
Repeated chars     : god
Lowercase          : wow this great
Keep not/no        : not
URL removal        : visit now


---

## Task 3 — Stress Testing on 10 Different Sentences

In [11]:
sample_inputs = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

# storing results here so I can reuse them in task 4 and 5
all_results = []

print("=" * 60)
print("STRESS TEST RESULTS")
print("=" * 60)

for idx, sentence in enumerate(sample_inputs, 1):
    tokens, clean = preprocess_text(sentence)
    all_results.append({"original": sentence, "tokens": tokens, "clean": clean})

    print(f"\nSentence {idx}:")
    print(f"  Original : {sentence}")
    print(f"  Tokens   : {tokens}")
    print(f"  Cleaned  : {clean}")

print("\n" + "=" * 60)

STRESS TEST RESULTS

Sentence 1:
  Original : Get 100% FREE access now!!!
  Tokens   : ['get', 'free', 'access', 'now']
  Cleaned  : get free access now

Sentence 2:
  Original : I absolutely looooved this product 😍😍
  Tokens   : ['absolutely', 'loved', 'this', 'product']
  Cleaned  : absolutely loved this product

Sentence 3:
  Original : Worst service ever... 0/10
  Tokens   : ['worst', 'service', 'ever']
  Cleaned  : worst service ever

Sentence 4:
  Original : Call me at 9876543210
  Tokens   : ['call']
  Cleaned  : call

Sentence 5:
  Original : This is THE best course!!!
  Tokens   : ['this', 'the', 'best', 'course']
  Cleaned  : this the best course

Sentence 6:
  Original : Visit https://openai.com now!
  Tokens   : ['visit', 'now']
  Cleaned  : visit now

Sentence 7:
  Original : Nooooo this is baaad!!!
  Tokens   : ['no', 'this', 'bad']
  Cleaned  : no this bad

Sentence 8:
  Original : OK OK OK I got it
  Tokens   : ['got']
  Cleaned  : got

Sentence 9:
  Original : Win $$$ 

---

## Task 4 — Token Analytics

In [12]:
def get_token_stats(tokens):
    # computing 3 basic stats for a list of tokens
    total = len(tokens)
    unique = len(set(tokens))

    # avg length — using 0 as fallback if token list is empty (edge case)
    if total > 0:
        avg_len = round(sum(len(w) for w in tokens) / total, 2)
    else:
        avg_len = 0

    return total, unique, avg_len


print("TOKEN ANALYTICS")
print("-" * 65)
print(f"{'#':<4} {'Sentence (short)':<35} {'Total':>6} {'Unique':>7} {'AvgLen':>7}")
print("-" * 65)

stats_per_sentence = []

for i, res in enumerate(all_results, 1):
    total, unique, avg_len = get_token_stats(res["tokens"])
    stats_per_sentence.append((total, unique, avg_len))

    # truncating the original sentence so the table stays neat
    short = res["original"][:32] + "..." if len(res["original"]) > 32 else res["original"]
    print(f"{i:<4} {short:<35} {total:>6} {unique:>7} {avg_len:>7}")

print("-" * 65)

TOKEN ANALYTICS
-----------------------------------------------------------------
#    Sentence (short)                     Total  Unique  AvgLen
-----------------------------------------------------------------
1    Get 100% FREE access now!!!              4       4     4.0
2    I absolutely looooved this produ...      4       4     6.5
3    Worst service ever... 0/10               3       3    5.33
4    Call me at 9876543210                    1       1     4.0
5    This is THE best course!!!               4       4    4.25
6    Visit https://openai.com now!            2       2     4.0
7    Nooooo this is baaad!!!                  3       3     3.0
8    OK OK OK I got it                        1       1     3.0
9    Win $$$ now!!! Limited offer!!!          4       4     4.5
10   I am not happy with this                 4       4     4.0
-----------------------------------------------------------------


In [13]:
# figuring out which sentence had the most noise
# I'm defining noise as: how many original words got dropped after cleaning
# more words lost = more noise

noise_scores = []
for i, res in enumerate(all_results):
    original_word_count = len(res["original"].split())
    cleaned_token_count = stats_per_sentence[i][0]
    noise_scores.append(original_word_count - cleaned_token_count)

most_noisy = noise_scores.index(max(noise_scores))

# sentence with most tokens after cleaning = most meaningful content retained
most_meaningful = max(range(len(stats_per_sentence)), key=lambda i: stats_per_sentence[i][0])

print(f"Most noisy sentence      -> Sentence {most_noisy + 1}")
print(f"  '{all_results[most_noisy]['original']}'")
print(f"  ({noise_scores[most_noisy]} words removed during cleaning)")
print()
print(f"Most meaningful retained -> Sentence {most_meaningful + 1}")
print(f"  '{all_results[most_meaningful]['original']}'")
print(f"  ({stats_per_sentence[most_meaningful][0]} tokens kept)")

Most noisy sentence      -> Sentence 8
  'OK OK OK I got it'
  (5 words removed during cleaning)

Most meaningful retained -> Sentence 1
  'Get 100% FREE access now!!!'
  (4 tokens kept)


---

## Task 5 — Frequency Analysis

In [14]:
from collections import Counter

# flattening all token lists from every sentence into one big list
combined_tokens = []
for res in all_results:
    combined_tokens.extend(res["tokens"])

freq = Counter(combined_tokens)

print(f"Total tokens across all sentences : {len(combined_tokens)}")
print(f"Unique tokens                     : {len(freq)}")
print()

# top 10 most common
print("Top 10 most frequent words:")
print("-" * 30)
for rank, (word, count) in enumerate(freq.most_common(10), 1):
    # simple bar chart using block characters so output is readable
    bar = "█" * count
    print(f"  {rank:>2}. {word:<15} {count}  {bar}")

print()

# bottom 5 least common
# most_common() returns highest first so reversing and taking last 5
print("Top 5 least frequent words:")
print("-" * 30)
least = freq.most_common()[:-6:-1]
for word, count in reversed(least):
    print(f"  {word:<15} {count}")

Total tokens across all sentences : 30
Unique tokens                     : 25

Top 10 most frequent words:
------------------------------
   1. this            4  ████
   2. now             3  ███
   3. get             1  █
   4. free            1  █
   5. access          1  █
   6. absolutely      1  █
   7. loved           1  █
   8. product         1  █
   9. worst           1  █
  10. service         1  █

Top 5 least frequent words:
------------------------------
  limited         1
  offer           1
  not             1
  happy           1
  with            1


---

## Task 6 — Full Pipeline Function

In [15]:
def full_pipeline(text_list):
    """
    Runs the entire preprocessing pipeline on a list of sentences.

    Input  : list of raw strings
    Output : dict with 'tokens' (list of token lists) and 'clean_sentences' (list of strings)
    """

    # basic guard — if someone passes something that isnt a list, we return empty
    if not isinstance(text_list, list) or len(text_list) == 0:
        print("please pass a non-empty list of strings")
        return {"tokens": [], "clean_sentences": []}

    tokens_list = []
    sentences_list = []

    for text in text_list:
        toks, sent = preprocess_text(text)
        tokens_list.append(toks)
        sentences_list.append(sent)

    return {
        "tokens": tokens_list,
        "clean_sentences": sentences_list
    }


# running the pipeline on our 10 sample sentences
output = full_pipeline(sample_inputs)

print("PIPELINE OUTPUT")
print("=" * 55)
for i, (toks, sent) in enumerate(zip(output["tokens"], output["clean_sentences"]), 1):
    print(f"\n[{i}]")
    print(f"  tokens   : {toks}")
    print(f"  sentence : {sent}")

PIPELINE OUTPUT

[1]
  tokens   : ['get', 'free', 'access', 'now']
  sentence : get free access now

[2]
  tokens   : ['absolutely', 'loved', 'this', 'product']
  sentence : absolutely loved this product

[3]
  tokens   : ['worst', 'service', 'ever']
  sentence : worst service ever

[4]
  tokens   : ['call']
  sentence : call

[5]
  tokens   : ['this', 'the', 'best', 'course']
  sentence : this the best course

[6]
  tokens   : ['visit', 'now']
  sentence : visit now

[7]
  tokens   : ['no', 'this', 'bad']
  sentence : no this bad

[8]
  tokens   : ['got']
  sentence : got

[9]
  tokens   : ['win', 'now', 'limited', 'offer']
  sentence : win now limited offer

[10]
  tokens   : ['not', 'happy', 'with', 'this']
  sentence : not happy with this


---

## Task 7 — Error Handling

In [16]:
# testing weird/broken inputs to make sure the function doesnt crash

edge_cases = [
    ("",           "empty string"),
    ("    ",       "only spaces"),
    ("😍🔥💯🎉",   "only emojis"),
    ("1234567890", "only numbers"),
    ("!!! ### $$$","only special chars"),
    (None,         "None value"),
    (42,           "integer instead of string"),
    (["hello"],    "list instead of string"),
]

print("EDGE CASE / ERROR HANDLING TESTS")
print("=" * 55)

for raw, label in edge_cases:
    try:
        tokens, sentence = preprocess_text(raw)
        # if it returned empty thats fine — means input had nothing usable
        if tokens == [] and sentence == "":
            print(f"  [{label}]  ->  returned empty (handled gracefully) ✓")
        else:
            print(f"  [{label}]  ->  tokens: {tokens}")
    except Exception as err:
        # shouldnt reach here but catching just in case
        print(f"  [{label}]  ->  ERROR: {err}")

print("=" * 55)
print("all edge cases handled without crashing")

EDGE CASE / ERROR HANDLING TESTS
  [empty string]  ->  returned empty (handled gracefully) ✓
  [only spaces]  ->  returned empty (handled gracefully) ✓
  [only emojis]  ->  returned empty (handled gracefully) ✓
  [only numbers]  ->  returned empty (handled gracefully) ✓
  [only special chars]  ->  returned empty (handled gracefully) ✓
  [None value]  ->  returned empty (handled gracefully) ✓
  [integer instead of string]  ->  returned empty (handled gracefully) ✓
  [list instead of string]  ->  returned empty (handled gracefully) ✓
all edge cases handled without crashing
